### Setup and Data Loading ###

In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# Load data and prep splits (exactly as before to ensure consistency)
df = pd.read_csv("../data/diabetes_cleaned.csv")
X = df.drop(columns=["Outcome"])
y = df["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data loaded and scaled successfully.")

Data loaded and scaled successfully.


### Part F - Naïve Bayes Experiment ###
For Gaussian Naïve Bayes, we will experiment with the var_smoothing hyperparameter by testing the default value, a smaller value, and a larger value

In [12]:
from sklearn.naive_bayes import GaussianNB

# Define variance smoothing parameters to test
smoothing_params = {
    "Default (1e-9)": 1e-9,
    "Smaller (1e-11)": 1e-11,
    "Larger (1e-1)": 1e-1
}

print("Naïve Bayes - Variance Smoothing Experiment:")
print("-" * 45)

for label, var_smoothing in smoothing_params.items():
    nb = GaussianNB(var_smoothing=var_smoothing)
    nb.fit(X_train_scaled, y_train)
    
    y_pred = nb.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    
    print(f"{label.ljust(18)} : Accuracy = {acc:.4f}")

Naïve Bayes - Variance Smoothing Experiment:
---------------------------------------------
Default (1e-9)     : Accuracy = 0.7273
Smaller (1e-11)    : Accuracy = 0.7273
Larger (1e-1)      : Accuracy = 0.7208


### Part G - Decision Tree Experiment ###
Building a shallow tree, a medium-depth tree, and a deep tree to demonstrate how depth affects training vs. testing performance (overfitting).

In [13]:
from sklearn.tree import DecisionTreeClassifier

# Define tree depths to test
tree_depths = {
    "Shallow Tree (max_depth=3)": 3,
    "Medium Tree (max_depth=7)": 7,
    "Deep Tree (max_depth=None)": None
}

print("Decision Tree - Depth and Overfitting Experiment:")
print("-" * 55)

for label, depth in tree_depths.items():
    # Initialize and train
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt.fit(X_train_scaled, y_train) # Trees don't strictly require scaling, but it's fine here
    
    # Evaluate on both Train and Test to check for overfitting
    train_acc = accuracy_score(y_train, dt.predict(X_train_scaled))
    test_acc = accuracy_score(y_test, dt.predict(X_test_scaled))
    
    print(f"{label}:")
    print(f"  -> Training Accuracy : {train_acc:.4f}")
    print(f"  -> Testing Accuracy  : {test_acc:.4f}\n")

Decision Tree - Depth and Overfitting Experiment:
-------------------------------------------------------
Shallow Tree (max_depth=3):
  -> Training Accuracy : 0.8811
  -> Testing Accuracy  : 0.8506

Medium Tree (max_depth=7):
  -> Training Accuracy : 0.9723
  -> Testing Accuracy  : 0.8506

Deep Tree (max_depth=None):
  -> Training Accuracy : 1.0000
  -> Testing Accuracy  : 0.8117



### Observations: Naïve Bayes & Decision Tree Tuning

#### 1. Naïve Bayes (Variance Smoothing)
* **Default vs. Smaller (1e-9 vs. 1e-11):** Both produced identical test accuracy (**72.73%**), indicating that smaller variance adjustments do not alter class boundary estimates for this dataset.
* **Larger Smoothing (1e-1):** Slightly degraded test accuracy to **72.08%**, as over-smoothing smooths feature variances too aggressively, dampening class separation capability.

#### 2. Decision Tree (Depth & Overfitting)
* **Shallow Tree (`max_depth=3`):** Achieved **85.06%** test accuracy with an 88.11% training accuracy. The small gap indicates strong generalization with minimal overfitting.
* **Medium Tree (`max_depth=7`):** Reached 97.23% training accuracy but maintained the same test accuracy (**85.06%**).
* **Deep Tree (`max_depth=None`):** Suffered from classic **overfitting**. It achieved a perfect **100% training accuracy** (memorizing noise), but test accuracy dropped significantly to **81.17%**.
* **Takeaway:** Restricting tree depth is essential to prevent overfitting on medical tabular data.

### Part H - Support Vector Machine (SVM) Experiment ###
We evaluate Linear vs. RBF kernels across various penalty values ($C$) and gamma choices ($\gamma$).  

In [14]:
from sklearn.svm import SVC

# SVM Experiments
svm_configs = [
    {"name": "Linear SVM (C=0.1)", "kernel": "linear", "C": 0.1, "gamma": "scale"},
    {"name": "Linear SVM (C=1.0)", "kernel": "linear", "C": 1.0, "gamma": "scale"},
    {"name": "RBF SVM (C=1.0, gamma=scale)", "kernel": "rbf", "C": 1.0, "gamma": "scale"},
    {"name": "RBF SVM (C=10.0, gamma=scale)", "kernel": "rbf", "C": 10.0, "gamma": "scale"},
    {"name": "RBF SVM (C=1.0, gamma=0.1)", "kernel": "rbf", "C": 1.0, "gamma": 0.1},
]

print("Support Vector Machine (SVM) Experiment:")
print("-" * 55)

for cfg in svm_configs:
    svm = SVC(kernel=cfg["kernel"], C=cfg["C"], gamma=cfg["gamma"], random_state=42)
    svm.fit(X_train_scaled, y_train)
    acc = accuracy_score(y_test, svm.predict(X_test_scaled))
    print(f"{cfg['name'].ljust(35)} : Test Accuracy = {acc:.4f}")

Support Vector Machine (SVM) Experiment:
-------------------------------------------------------
Linear SVM (C=0.1)                  : Test Accuracy = 0.7403
Linear SVM (C=1.0)                  : Test Accuracy = 0.7403
RBF SVM (C=1.0, gamma=scale)        : Test Accuracy = 0.8377
RBF SVM (C=10.0, gamma=scale)       : Test Accuracy = 0.8442
RBF SVM (C=1.0, gamma=0.1)          : Test Accuracy = 0.8377


 ### Part I - Logistic Regression Experiment ####
 We test L1 (Lasso) vs. L2 (Ridge) penalties across regularisation strengths ($C$).

In [15]:
from sklearn.linear_model import LogisticRegression

# Modern Logistic Regression configurations
# penalty='elasticnet' with solver='saga' uses l1_ratio:
# l1_ratio = 1.0 -> Pure L1 (Lasso)
# l1_ratio = 0.0 -> Pure L2 (Ridge)
lr_configs = [
    {"name": "L1 Regularization (C=0.1)", "l1_ratio": 1.0, "C": 0.1},
    {"name": "L1 Regularization (C=1.0)", "l1_ratio": 1.0, "C": 1.0},
    {"name": "L2 Regularization (C=0.1)", "l1_ratio": 0.0, "C": 0.1},
    {"name": "L2 Regularization (C=1.0)", "l1_ratio": 0.0, "C": 1.0},
    {"name": "L2 Regularization (C=10.0)", "l1_ratio": 0.0, "C": 10.0},
]

print("Logistic Regression Experiment (Modern Syntax):")
print("-" * 55)

for cfg in lr_configs:
    lr = LogisticRegression(
        penalty="elasticnet",
        l1_ratio=cfg["l1_ratio"],
        C=cfg["C"],
        solver="saga",
        max_iter=5000,  # SAGA solver requires higher iterations for convergence
        random_state=42
    )
    lr.fit(X_train_scaled, y_train)
    acc = accuracy_score(y_test, lr.predict(X_test_scaled))
    print(f"{cfg['name'].ljust(35)} : Test Accuracy = {acc:.4f}")

Logistic Regression Experiment (Modern Syntax):
-------------------------------------------------------
L1 Regularization (C=0.1)           : Test Accuracy = 0.7208
L1 Regularization (C=1.0)           : Test Accuracy = 0.7078
L2 Regularization (C=0.1)           : Test Accuracy = 0.7078
L2 Regularization (C=1.0)           : Test Accuracy = 0.7078
L2 Regularization (C=10.0)          : Test Accuracy = 0.7078


c:\Users\User\Desktop\samsung-ml-regression-tasks\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\User\Desktop\samsung-ml-regression-tasks\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn

### Part J - K-Nearest Neighbours (KNN) Experiment ###
 We test neighbor counts ($K \in [3, 5, 7, 9, 11, 15]$) and weight functions (uniform vs distance).

In [16]:
from sklearn.neighbors import KNeighborsClassifier

k_values = [3, 5, 7, 9, 11, 15]
weights_list = ["uniform", "distance"]

print("K-Nearest Neighbours (KNN) Experiment:")
print("-" * 55)

for w in weights_list:
    print(f"Weight Type: {w.upper()}")
    for k in k_values:
        knn = KNeighborsClassifier(n_neighbors=k, weights=w)
        knn.fit(X_train_scaled, y_train)
        acc = accuracy_score(y_test, knn.predict(X_test_scaled))
        print(f"  K = {str(k).ljust(2)} : Test Accuracy = {acc:.4f}")
    print()

K-Nearest Neighbours (KNN) Experiment:
-------------------------------------------------------
Weight Type: UNIFORM
  K = 3  : Test Accuracy = 0.7727
  K = 5  : Test Accuracy = 0.8117
  K = 7  : Test Accuracy = 0.8052
  K = 9  : Test Accuracy = 0.7987
  K = 11 : Test Accuracy = 0.8052
  K = 15 : Test Accuracy = 0.7727

Weight Type: DISTANCE
  K = 3  : Test Accuracy = 0.7727
  K = 5  : Test Accuracy = 0.8117
  K = 7  : Test Accuracy = 0.8117
  K = 9  : Test Accuracy = 0.8052
  K = 11 : Test Accuracy = 0.8052
  K = 15 : Test Accuracy = 0.7792



### Part K - Random Forest Experiment
We vary tree counts (n_estimators) and tree depths (max_depth)

In [17]:
from sklearn.ensemble import RandomForestClassifier

rf_configs = [
    {"name": "RF (n=50, depth=3)", "n_estimators": 50, "max_depth": 3},
    {"name": "RF (n=100, depth=3)", "n_estimators": 100, "max_depth": 3},
    {"name": "RF (n=100, depth=7)", "n_estimators": 100, "max_depth": 7},
    {"name": "RF (n=200, depth=None)", "n_estimators": 200, "max_depth": None},
]

print("Random Forest Experiment:")
print("-" * 55)

for cfg in rf_configs:
    rf = RandomForestClassifier(
        n_estimators=cfg["n_estimators"],
        max_depth=cfg["max_depth"],
        random_state=42
    )
    rf.fit(X_train_scaled, y_train)
    acc = accuracy_score(y_test, rf.predict(X_test_scaled))
    print(f"{cfg['name'].ljust(30)} : Test Accuracy = {acc:.4f}")


Random Forest Experiment:
-------------------------------------------------------
RF (n=50, depth=3)             : Test Accuracy = 0.8636
RF (n=100, depth=3)            : Test Accuracy = 0.8571
RF (n=100, depth=7)            : Test Accuracy = 0.8636
RF (n=200, depth=None)         : Test Accuracy = 0.8636


### Summary of Model Tuning Experiments

* **KNN:** Peak accuracy reached at $K=5$ (**81.17%**). Distance-based weighting provided slight performance gains at higher neighbor counts.
* **Random Forest:** Exceptional stability across all configurations (**86.36%**). Restricting max depth to 3 prevents overfitting while maximizing generalization accuracy.